In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import joblib

In [3]:

def load_all_tests(base_dir='../hasil_eksperimenV3_jw/single',
                   rel_path=('ml_training', 'ml_model', 'qem_model_1.pkl')):
    """
    Membaca semua predicted_data_1.csv dari folder test_*
    dan mengembalikan satu DataFrame besar dengan kolom 'dataset'
    sebagai penanda asal folder (test_01, test_02, dst).
    """
    dataframes = []

    for test_dir in sorted(os.listdir(base_dir)):

        path = os.path.join(base_dir, test_dir, *rel_path)
        if os.path.exists(path):
            dataframes.append({
                'dataset' : test_dir,
                'model': joblib.load(path)
                })
        else:
            print(f"Peringatan: file tidak ditemukan: {path}")

    if not dataframes:
        raise FileNotFoundError(
            "Tidak ada file yang ditemukan di folder test_*"
        )

    return dataframes

# Contoh pemakaian:
df_all = load_all_tests()

In [4]:
model = df_all[1]['model']

In [5]:
model.feature_names_in_

array(['noisy_energy', 'param_0', 'param_1', 'param_2', 'param_3',
       'param_4', 'param_5', 'param_6', 'param_7', 'param_8', 'param_9',
       'param_10', 'param_11', 'obs0I', 'obs0X', 'obs0Y', 'obs0Z',
       'obs1I', 'obs1X', 'obs1Y', 'obs1Z', 'obs2I', 'obs2X', 'obs2Y',
       'obs2Z', 'obs3I', 'obs3X', 'obs3Y', 'obs3Z'], dtype=object)

In [21]:
import pandas as pd

# Ambil data dari model Anda
names = model.feature_names_in_
importances = model.feature_importances_

# Buat DataFrame
feat_importances = pd.DataFrame({
    'Feature Name': names,
    'Importance Score': importances
})

# Urutkan dari nilai tertinggi ke terendah
feat_importances = feat_importances.sort_values(by='Importance Score', ascending=False).reset_index(drop=True)

# Tampilkan (Jika di Jupyter Notebook, gunakan display() atau langsung panggil variabelnya)
print(feat_importances.head())

   Feature Name  Importance Score
0  noisy_energy          0.997446
1       param_4          0.000230
2       param_3          0.000230
3       param_2          0.000225
4       param_8          0.000219


In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1. Memuat Data
file_path = '../unified_output_v3/single_02/ml_training/train_data/train_data_1.csv'
df = pd.read_csv(file_path)

# 2. Preprocessing Data
# Target adalah 'ideal_energy'
target_col = 'ideal_energy'

# Memisahkan Fitur (X) dan Target (y)
X_raw = df.drop(columns=[target_col])
y = df[target_col]

# Melakukan One-Hot Encoding pada kolom 'observable'
# Menggunakan pd.get_dummies yang otomatis mendeteksi kolom kategorikal jika dispesifikasikan,
# atau kita bisa eksplisit menunjuk kolomnya.
X = pd.get_dummies(X_raw, columns=['observable'])

# 3. Split Data (Train-Test Split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Definisi dan Pelatihan Model
# (Saya menggunakan Random Forest sebagai contoh model yang memiliki feature_importances_)
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

# 5. Menghitung Metrik
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

# 6. Mengambil Feature Importance
importances = model.feature_importances_
feature_names = model.feature_names_in_ # Nama fitur setelah one-hot encoding

df_importance = pd.DataFrame({
    'Fitur': feature_names,
    'Importance Score': importances
})

# Mengurutkan dari yang paling berpengaruh
df_importance = df_importance.sort_values(by='Importance Score', ascending=False).reset_index(drop=True)

# 7. Menyusun Metrik Akhir
metrics = {
    "train_mse": train_mse,
    "test_mse": test_mse,
    "train_r2": train_r2,
    "test_r2": test_r2,
    "n_samples": len(X),
    "n_features": X.shape[1]
}

# --- MENAMPILKAN HASIL ---
print("=== Tabel Fitur Paling Berpengaruh (Top 10) ===")
print(df_importance.head(10))

print("\n=== Ringkasan Metrik Model ===")
df_metrics = pd.DataFrame([metrics])
print(df_metrics)

=== Tabel Fitur Paling Berpengaruh (Top 10) ===
          Fitur  Importance Score
0  noisy_energy          0.999349
1       param_3          0.000058
2      param_11          0.000057
3       param_8          0.000053
4       param_1          0.000052
5       param_4          0.000051
6    zne_energy          0.000051
7       param_2          0.000051
8       param_0          0.000048
9      param_10          0.000048

=== Ringkasan Metrik Model ===
   train_mse  test_mse  train_r2   test_r2  n_samples  n_features
0   0.000042  0.000299  0.999853  0.998882       2000          15
